# MGSC 416 — Coffee shop inventory and ordering

## Section 1 — Introduction

This notebook supports an **MGSC 416** semester project on **data-driven operations** for a multi-site coffee business. We use real-world-style transaction data (Maven Roasters) to study **how much to stock** at each location and how demand varies over time.

### Context and motivation

Retail cafes must balance **service level** (having enough product when customers arrive) with **waste and tied-up cash** when prepared or perishable items are not sold the same day. Line-level sales history lets us recover **true baskets** (several products sold in one checkout) and **per-store** patterns, which a simple daily total cannot show.

### What this notebook does

- Load and validate **line-level** transactions from Excel.
- Build **`order_id`** from store + timestamp so lines sold in the same second count as one order.
- Summarise demand and revenue for **each store** and produce figures for the report.
- Later sections (stubs for now) will add calendar features, forecasting, and **optimisation** for order quantities.

### Data source

**Coffee Shop Sales (1)(Transactions).csv** — 149,116 transaction rows with timestamps, store, product hierarchy, quantities, prices, and pre-computed `product_type_sized` and `COGS_product_type` columns.

### Tools

Python **pandas**, **numpy**, **matplotlib**, **seaborn**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "Coffee Shop Sales (1)(Transactions).csv"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path exists:", DATA_PATH.exists())

---

## Section 2 — Data loading

Load the **Transactions** sheet, inspect shape, dtypes, and missing values.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

raw = pd.read_csv(DATA_PATH, parse_dates=['transaction_date'])
print("Shape:", raw.shape)
print(raw.dtypes)
raw.info()
display(raw.head())
missing = raw.isna().sum()
print("Missing per column:")
print(missing[missing > 0] if missing.any() else "None")

---

## Section 3 — Cleaning and validation (person 1)

Checklist: duplicates → `transaction_id` pattern → datetimes → numeric checks → product hierarchy → stores → **`order_id`** → cost columns → optional export.

In [ ]:
df = raw.copy()
n_before = len(df)
dup_mask = df.duplicated(keep=False)
n_dup = dup_mask.sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Full-row duplicates removed: {n_before - len(df)} rows (checked {n_dup} in duplicate groups)")

In [ ]:
tid_counts = df["transaction_id"].value_counts()
print("transaction_id unique:", df["transaction_id"].nunique(), "rows:", len(df))
print("Max rows sharing one transaction_id:", int(tid_counts.max()))
print(
    "Interpretation: if max is 1, each row is its own id — use datetime basket key for orders."
)

In [ ]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"])
df["transaction_dt"] = pd.to_datetime(
    df["transaction_date"].dt.strftime("%Y-%m-%d")
    + " "
    + df["transaction_time"].astype(str)
)
df["date"] = df["transaction_dt"].dt.normalize()
df["hour"] = df["transaction_dt"].dt.hour
df["dow"] = df["transaction_dt"].dt.dayofweek
df["month"] = df["transaction_dt"].dt.month
df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
df.head()

In [ ]:
bad_qty = df["transaction_qty"] <= 0
bad_price = df["unit_price"] < 0
print("Rows with qty <= 0:", bad_qty.sum())
print("Rows with unit_price < 0:", bad_price.sum())
df = df.loc[~(bad_qty | bad_price)].copy()


### Numeric checks, then revenue and cost columns

After dropping invalid quantities and prices, we add **line revenue** and a **per-unit COGS** used later for waste. **Working assumption:** **60% gross margin** on the selling price, so **COGS = 40% of `unit_price`**. If a prepared unit does **not** sell that day, we treat the money at risk as that COGS (**`unsold_loss_per_unit`**), i.e. **0.4 × unit_price** for each unit written off.

You can change `GROSS_MARGIN` below for sensitivity analysis; downstream optimisation should read the same constants.


In [ ]:
# Margin assumption for same-day unsold units (inventory write-off)
GROSS_MARGIN = 0.60
COGS_SHARE = 1.0 - GROSS_MARGIN

df["line_revenue"] = df["transaction_qty"] * df["unit_price"]
df["unit_cogs"] = COGS_SHARE * df["unit_price"]
df["unsold_loss_per_unit"] = df["unit_cogs"]

print(f"Gross margin assumption: {GROSS_MARGIN:.0%} of shelf price")
print(f"COGS share (economic loss if one unit is not sold that day): {COGS_SHARE:.0%} of unit_price")


In [ ]:
for col in ["product_category", "product_type", "product_detail"]:
    df[col] = df[col].fillna("unknown")
n_unknown = (df[["product_category", "product_type", "product_detail"]] == "unknown").any(axis=1).sum()
print("Rows with any unknown product field:", n_unknown)

In [ ]:
stores = df[["store_id", "store_location"]].drop_duplicates().sort_values("store_id")
display(stores)
assert df["store_id"].nunique() == 3, "Expected three stores"
print("Store check: OK (3 locations).")

In [ ]:
df["order_key"] = (
    df["store_id"].astype(str)
    + "|"
    + df["transaction_dt"].dt.strftime("%Y-%m-%d %H:%M:%S")
)
df["order_id"] = df["order_key"]
lines_per_tid = df.groupby("transaction_id", observed=True).size()
lines_per_order = df.groupby("order_id", observed=True).size()
print("Lines per transaction_id (max):", int(lines_per_tid.max()))
print("Lines per order_id (max):", int(lines_per_order.max()))
print("Multi-line orders:", int((lines_per_order > 1).sum()))

In [ ]:
lines_path = OUTPUTS_DIR / "lines_clean.csv"
df.to_csv(lines_path, index=False)
print("Wrote:", lines_path, "rows:", len(df))

---

## Section 4 — Order-level dataset

One row per `order_id` with basket size and revenue (for AOV and multi-item behaviour).

In [ ]:
order_agg = (
    df.groupby(["order_id", "store_id", "store_location", "date"], observed=True)
    .agg(
        n_lines=("transaction_id", "count"),
        order_qty_total=("transaction_qty", "sum"),
        order_revenue=("line_revenue", "sum"),
    )
    .reset_index()
)
orders_path = OUTPUTS_DIR / "orders_clean.csv"
order_agg.to_csv(orders_path, index=False)
print("Orders:", len(order_agg), "→", orders_path)
order_agg.describe()

---

## Section 4.5 — Outlier checks

Before feature engineering, we **screen** numeric columns for extreme values. Goals:

- See tails of **`unit_price`**, **`transaction_qty`**, and **`line_revenue`** (and order totals).
- Flag rows outside **1.5×IQR** fences (standard boxplot rule) for prices and line revenue — report counts, do **not** auto-drop unless the team agrees (document in the report).
- Spot **quantity spikes** relative to the distribution (e.g. very high `transaction_qty`).
- Check **price consistency** per `product_detail` (coefficient of variation) where the same SKU has multiple price points.

Plots are saved to **`outputs/`** for the appendix.


In [ ]:
num_cols = ["unit_price", "transaction_qty", "line_revenue"]
print("Numeric summary (with selected percentiles):")
display(df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))


def iqr_fences(s: pd.Series, k: float = 1.5) -> tuple[float, float]:
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return float(q1 - k * iqr), float(q3 + k * iqr)


for col in ["unit_price", "line_revenue"]:
    lo, hi = iqr_fences(df[col])
    mask = (df[col] < lo) | (df[col] > hi)
    print(
        f"{col}: IQR fences [{lo:.4g}, {hi:.4g}] — outside: {mask.sum():,} rows ({100 * mask.mean():.2f}%)"
    )

qty_threshold = max(20.0, float(df["transaction_qty"].quantile(0.995)))
mask_qty = df["transaction_qty"] >= qty_threshold
print(
    f"\ntransaction_qty >= {qty_threshold:.0f} (high-quantity tail): {mask_qty.sum():,} rows"
)
if mask_qty.any():
    display(df.loc[mask_qty, num_cols + ["product_detail", "store_location"]].head(15))

print("\nTop 15 lines by line_revenue (inspect for data quirks):")
display(
    df.nlargest(15, "line_revenue")[
        ["store_location", "product_detail", "transaction_qty", "unit_price", "line_revenue", "date"]
    ]
)

price_by_sku = (
    df.groupby("product_detail", observed=True)["unit_price"]
    .agg(count="count", mean="mean", std="std", min="min", max="max")
    .reset_index()
)
price_by_sku = price_by_sku[price_by_sku["count"] >= 30]
price_by_sku["cv"] = price_by_sku["std"] / price_by_sku["mean"].replace(0, np.nan)
print("\nproduct_detail with highest price CV (min 30 lines) — possible inconsistencies:")
display(price_by_sku.sort_values("cv", ascending=False).head(12))

print("\nLargest basket totals (order_revenue):")
display(
    order_agg.nlargest(12, "order_revenue")[
        ["store_location", "date", "n_lines", "order_qty_total", "order_revenue"]
    ]
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols, strict=True):
    cap = df[col].quantile(0.99)
    clipped = df[col].clip(upper=cap)
    clipped.hist(bins=50, ax=ax, color="steelblue", edgecolor="white", alpha=0.85)
    ax.set_title(f"{col} (clipped at 99th pct for display)")
    ax.set_ylabel("Count")
fig.suptitle("Distribution of key numerics (tails clipped for plotting)")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_outliers_numeric_hist.png", bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 4))
df.boxplot(column="unit_price", by="product_category", ax=ax2, rot=45)
ax2.set_title("unit_price by product_category")
ax2.set_xlabel("product_category")
plt.suptitle("")
fig2.tight_layout()
fig2.savefig(OUTPUTS_DIR / "fig_outliers_price_by_category.png", bbox_inches="tight")
plt.show()


---

## Section 5 — Feature engineering (person 2)

After outlier screening (Section 4.5), we extend the cleaned line-level `DataFrame` with **calendar fields**, **US (federal + New York state) holiday** flags from `us_ny_holidays_2023.csv` (aligned with Maven Roasters' NYC-area stores), simple **product tags** for regression or interaction terms, and an **enriched export** for Sections 7–8.

The **first** code cell covers **5.1** extra calendar columns, **5.2** `us_ny_holidays_2023.csv` merge, **5.3** drink / food coarse tags, **5.4** write `outputs/lines_enriched.csv`. **5.5** (following cells) runs **descriptive analysis** on temporal and holiday fields.

**Definitions:** `transaction_dt` is **calendar date** from the file plus **`transaction_time`** (hour-minute-second of the line). **`date`** is that instant normalised to **midnight** for daily joins and holiday matching. **`hour`** / **`dow`** / **`month`** follow `transaction_dt`. **`is_holiday`** is 1 when `date` matches a row in `us_ny_holidays_2023.csv` (filtered to years present in the transactions). **`days_to_next_holiday`** is whole days from that calendar day until the **next** holiday date in the filtered calendar **on or after** that day (0 on a holiday); it is computed once per distinct `date` then mapped (fast and consistent).

**Note:** `date`, `hour`, `dow`, `month`, and `is_weekend` are already created in Section 3; we add week, season, month boundaries, and labels here.


In [ ]:
# --- 5.1 Calendar features (beyond Section 3) ---
df["day_name"] = df["date"].dt.day_name()
df["year"] = df["date"].dt.year
# Handle occasional NaT values safely before integer conversion
week_iso = df["date"].dt.isocalendar().week
df["week"] = week_iso.fillna(0).astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)


def season_from_month(m: int) -> str:
    if m in (12, 1, 2):
        return "Winter"
    if m in (3, 4, 5):
        return "Spring"
    if m in (6, 7, 8):
        return "Summer"
    return "Autumn"


df["season"] = df["month"].map(season_from_month)

# --- 5.2 Holidays (US federal + NY state — 2023 list, NYC-relevant) ---
HOLIDAYS_PATH = PROJECT_ROOT / "us_ny_holidays_2023.csv"

if HOLIDAYS_PATH.exists():
    holidays_cal = pd.read_csv(HOLIDAYS_PATH, parse_dates=["Date"])
    holidays_cal = holidays_cal.rename(
        columns={"Date": "date", "Holiday": "holiday", "Type": "holiday_type"}
    )
else:
    print(f"Holiday file not found at {HOLIDAYS_PATH}. Proceeding with no holiday flags.")
    holidays_cal = pd.DataFrame(columns=["date", "holiday", "holiday_type"])
    holidays_cal["date"] = pd.to_datetime(holidays_cal["date"])
years_in_df = sorted(df["date"].dropna().dt.year.unique())
holidays_y = holidays_cal[holidays_cal["date"].dt.year.isin(years_in_df)].copy()
holidays_y["date_norm"] = holidays_y["date"].dt.normalize()
holiday_dates = set(holidays_y["date_norm"].dt.date)
df["is_holiday"] = df["date"].dt.date.isin(holiday_dates).astype(int)

holiday_ts_sorted = sorted(pd.to_datetime(holidays_y["date_norm"].unique()))


def days_to_next_holiday(d, holiday_list):
    d_norm = pd.Timestamp(d).normalize()
    future = [h for h in holiday_list if h >= d_norm]
    if not future:
        return np.nan
    return (future[0] - d_norm).days


# One lookup per distinct calendar date (same result as row-wise apply, much faster)
_date_norm_series = df["date"].dt.normalize()
_valid_norm = _date_norm_series.dropna()
uniq_norm = sorted(_valid_norm.unique(), key=lambda x: pd.Timestamp(x))
_days_map = {
    pd.Timestamp(d).normalize(): days_to_next_holiday(d, holiday_ts_sorted) for d in uniq_norm
}
df["days_to_next_holiday"] = _date_norm_series.map(
    lambda x: _days_map.get(pd.Timestamp(x).normalize(), np.nan) if pd.notna(x) else np.nan
)

print("Years in transactions:", years_in_df)
print(f"Holiday calendar rows (filtered years): {len(holidays_y)}")
print(f"Lines on holidays: {df['is_holiday'].sum():,}")
print(
    "days_to_next_holiday min–max:",
    df["days_to_next_holiday"].min(),
    "–",
    df["days_to_next_holiday"].max(),
)

# --- 5.3 Product tags (Maven Roasters `product_category`) ---
DRINK_CATEGORIES = {
    "Coffee",
    "Tea",
    "Drinking Chocolate",
    "Loose Tea",
    "Coffee beans",
    "Flavours",
}
HOT_DRINK_CATEGORIES = {"Coffee", "Tea", "Drinking Chocolate", "Loose Tea"}
FOODISH_CATEGORIES = {"Bakery", "Packaged Chocolate", "Branded"}

df["is_drink"] = df["product_category"].isin(DRINK_CATEGORIES).astype(int)
df["is_hot_drink"] = df["product_category"].isin(HOT_DRINK_CATEGORIES).astype(int)
df["item_category_coarse"] = np.where(
    df["product_category"].isin(FOODISH_CATEGORIES),
    "food_merch",
    np.where(df["is_drink"] == 1, "drinks", "other"),
)

print("\nitem_category_coarse:")
print(df["item_category_coarse"].value_counts())
print("Hot drink lines:", f"{df['is_hot_drink'].sum():,}")

# --- 5.4 Export enriched line-level table ---
enriched_path = OUTPUTS_DIR / "lines_enriched.csv"
df.to_csv(enriched_path, index=False)
print("\nWrote:", enriched_path, "shape:", df.shape)
display(
    df[
        [
            "date",
            "store_location",
            "is_holiday",
            "days_to_next_holiday",
            "season",
            "item_category_coarse",
            "is_hot_drink",
        ]
    ].head()
)



### 5.5 Descriptive analysis — temporal and holiday features

Tables and plots below summarise each engineered time and holiday column. **Assumption:** store timestamps are treated as a single timezone (no offset column in the source file).


In [ ]:
# --- 5.5 Descriptive analysis: temporal + holiday + product-tag fields ---
temporal_num = [
    "hour",
    "dow",
    "month",
    "year",
    "week",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "is_holiday",
    "days_to_next_holiday",
    "is_drink",
    "is_hot_drink",
]
temporal_cat = ["day_name", "season", "item_category_coarse"]
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

print("=== Calendar span (line-level) ===")
print("transaction_dt min / max:", df["transaction_dt"].min(), "->", df["transaction_dt"].max())
print("Distinct calendar dates (date):", df["date"].nunique())
print("Year values in data:", sorted(df["year"].dropna().astype(int).unique().tolist()))

print("\n=== Numeric / binary temporal features — describe ===")
display(df[temporal_num].describe().T)

print("\n=== hour (from transaction_dt) — line counts ===")
display(df["hour"].value_counts().sort_index().to_frame("n_lines"))

print("\n=== day of week (0=Mon) — line counts ===")
vc_dow = df["dow"].value_counts().sort_index()
vc_dow.index = [dow_labels[i] for i in vc_dow.index.astype(int)]
display(vc_dow.to_frame("n_lines"))

print("\n=== month — line counts ===")
display(df["month"].value_counts().sort_index().to_frame("n_lines"))

print("\n=== ISO week — range ===", int(df["week"].min()), "–", int(df["week"].max()))

print("\n=== Categorical labels — value counts ===")
for col in temporal_cat:
    print(f"\n-- {col} --")
    display(df[col].value_counts(dropna=False))

print("\n=== Holiday file rows used for transaction years (head) ===")
display(holidays_y[["date_norm", "holiday", "holiday_type"]].rename(columns={"date_norm": "date"}).head(20))

holiday_lookup = (
    holidays_y.sort_values("date_norm")
    .drop_duplicates(subset=["date_norm"], keep="first")
    .set_index("date_norm")["holiday"]
)
print("\n=== Lines on public holidays — by holiday name ===")
lines_on_h = df.loc[df["is_holiday"] == 1, ["date"]].copy()
lines_on_h["holiday_name"] = lines_on_h["date"].dt.normalize().map(holiday_lookup)
display(lines_on_h["holiday_name"].value_counts())

print("\n=== Calendar days in data that appear in us_ny_holidays_2023.csv ===")
data_days = {pd.Timestamp(d).normalize() for d in df["date"].dt.normalize().dropna().unique()}
cal_days = {pd.Timestamp(d).normalize() for d in pd.Series(holiday_lookup.index).dropna().unique()}
overlap_days = sorted(data_days & cal_days)
print(len(overlap_days), "days, sample:", [d.strftime("%Y-%m-%d") for d in overlap_days[:12]])

print("\n=== days_to_next_holiday — describe ===")
display(df["days_to_next_holiday"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))
nan_d = int(df["days_to_next_holiday"].isna().sum())
if nan_d:
    print("NaN rows:", nan_d, "(no holiday on or after that calendar day in filtered file)")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
df["hour"].hist(bins=range(25), ax=axes[0, 0], color="steelblue", edgecolor="white")
axes[0, 0].set_title("Hour of sale (transaction_dt)")
axes[0, 0].set_xlabel("Hour")
dow_counts = df["dow"].value_counts().sort_index().reindex(range(7), fill_value=0)
dow_counts.index = dow_labels
dow_counts.plot(kind="bar", ax=axes[0, 1], rot=0, color="0.55")
axes[0, 1].set_title("Lines by day of week")
axes[0, 1].set_xlabel("")
df["days_to_next_holiday"].dropna().hist(bins=40, ax=axes[1, 0], color="coral", edgecolor="white")
axes[1, 0].set_title("Days until next listed holiday")
axes[1, 0].set_xlabel("days_to_next_holiday")
df["season"].value_counts().plot(kind="barh", ax=axes[1, 1])
axes[1, 1].set_title("Season (from calendar month)")
fig.suptitle("Temporal and holiday feature distributions")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_descriptive_temporal_features.png", bbox_inches="tight")
plt.show()


---

## Section 6 — EDA visualisations (person 1)

Figures saved under **`outputs/`**, faceted by **`store_location`** where noted.

In [ ]:
store_order = [s for s in df["store_location"].unique()]
palette = sns.color_palette("muted", n_colors=len(store_order))
store_palette = dict(zip(store_order, palette, strict=True))

In [ ]:
daily_lines = (
    df.groupby(["date", "store_location"], observed=True).size().reset_index(name="n_lines")
)
daily_orders = (
    df.groupby(["date", "store_location"], observed=True)["order_id"]
    .nunique()
    .reset_index(name="n_orders")
)
daily_vol = daily_lines.merge(daily_orders, on=["date", "store_location"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, store in zip(axes, sorted(daily_vol["store_location"].unique()), strict=True):
    sub = daily_vol[daily_vol["store_location"] == store].sort_values("date")
    ax.plot(sub["date"], sub["n_lines"], label="Line items", color=store_palette[store])
    ax.plot(sub["date"], sub["n_orders"], label="Orders", linestyle="--", color="0.3")
    ax.set_title(store)
    ax.set_xlabel("Date")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
fig.suptitle("Daily line items vs distinct orders")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_daily_lines_vs_orders.png", bbox_inches="tight")
plt.show()

In [ ]:
daily_rev = (
    df.groupby(["date", "store_location"], observed=True)["line_revenue"]
    .sum()
    .reset_index()
    .sort_values(["store_location", "date"])
)
daily_rev["rev_7d"] = daily_rev.groupby("store_location", observed=True)["line_revenue"].transform(
    lambda s: s.rolling(7, min_periods=1).mean()
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, store in zip(axes, sorted(daily_rev["store_location"].unique()), strict=True):
    sub = daily_rev[daily_rev["store_location"] == store]
    ax.plot(sub["date"], sub["line_revenue"], alpha=0.35, label="Daily revenue")
    ax.plot(sub["date"], sub["rev_7d"], label="7-day mean", color=store_palette[store])
    ax.set_title(store)
    ax.set_xlabel("Date")
    ax.set_ylabel("Revenue")
    ax.legend(fontsize=8)
fig.suptitle("Daily revenue and 7-day rolling mean")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_daily_revenue_rolling.png", bbox_inches="tight")
plt.show()

In [ ]:
# Revenue heatmap by store, hour, and day of week (clean visual, no annotations)
heat = (
    df.groupby(["store_location", "dow", "hour"], observed=True)["line_revenue"]
    .sum()
    .reset_index()
)

dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
stores = sorted(heat["store_location"].unique())

# Build store pivots first so all panels share a consistent colour scale
store_pivots = {}
for store in stores:
    sub = heat[heat["store_location"] == store]
    pivot = sub.pivot(index="hour", columns="dow", values="line_revenue").reindex(
        index=range(24), columns=range(7), fill_value=0
    )
    pivot.columns = dow_labels
    store_pivots[store] = pivot

vmax = max(p.values.max() for p in store_pivots.values())

fig, axes = plt.subplots(1, len(stores), figsize=(16, 4.8), sharey=True)
if len(stores) == 1:
    axes = [axes]

for ax, store in zip(axes, stores, strict=True):
    pivot = store_pivots[store]
    sns.heatmap(
        pivot,
        cmap="YlOrRd",
        vmin=0,
        vmax=vmax,
        linewidths=0.25,
        linecolor="white",
        cbar_kws={"label": "Revenue"},
        ax=ax,
    )
    ax.set_title(store, fontsize=11, pad=8)
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Hour of day")
    ax.set_yticks([0.5, 6.5, 12.5, 18.5, 23.5])
    ax.set_yticklabels([0, 6, 12, 18, 23], rotation=0)

fig.suptitle("Revenue heatmap: hour x day of week", fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_heatmap_hour_dow.png", bbox_inches="tight")
plt.show()

# Table below: highest day of week by revenue share for each store
top_day_rows = []
for store, pivot in store_pivots.items():
    day_totals = pivot.sum(axis=0)
    top_day = day_totals.idxmax()
    top_revenue = float(day_totals.loc[top_day])
    total_revenue = float(day_totals.sum())
    share_pct = (top_revenue / total_revenue * 100) if total_revenue > 0 else np.nan
    top_day_rows.append(
        {
            "store_location": store,
            "top_day": top_day,
            "top_day_revenue": round(top_revenue, 2),
            "top_day_share_pct": round(share_pct, 2),
        }
    )

top_day_summary = pd.DataFrame(top_day_rows).sort_values("store_location").reset_index(drop=True)
print("Highest day by revenue share (per store):")
display(top_day_summary)


In [ ]:
N_TOP = 12
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
stores = sorted(df["store_location"].dropna().astype(str).unique())
for i, store in enumerate(stores):
    sub = df[df["store_location"].astype(str) == store]
    top_rev = (
        sub.groupby("product_detail", observed=True)["line_revenue"]
        .sum()
        .nlargest(N_TOP)
        .iloc[::-1]
    )
    top_qty = (
        sub.groupby("product_detail", observed=True)["transaction_qty"]
        .sum()
        .nlargest(N_TOP)
        .iloc[::-1]
    )
    axes[i, 0].barh(top_rev.index.astype(str), top_rev.values, color=store_palette[store])
    axes[i, 0].set_title(f"{store} — top {N_TOP} by revenue")
    axes[i, 1].barh(top_qty.index.astype(str), top_qty.values, color="0.5")
    axes[i, 1].set_title(f"{store} — top {N_TOP} by quantity")
fig.suptitle("Top products by store")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_top_products_rev_qty.png", bbox_inches="tight")
plt.show()

In [ ]:
mix = (
    df.groupby(["store_location", "month", "product_category"], observed=True)["line_revenue"]
    .sum()
    .reset_index()
)
mix_pivot = mix.pivot_table(
    index=["store_location", "month"],
    columns="product_category",
    values="line_revenue",
    fill_value=0,
)
mix_pct = mix_pivot.div(mix_pivot.sum(axis=1), axis=0)

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for ax, store in zip(axes, sorted(mix_pct.index.get_level_values(0).unique()), strict=True):
    sub = mix_pct.xs(store, level=0)
    sub.plot(kind="bar", stacked=True, ax=ax, width=0.85, legend=(store == sorted(mix_pct.index.get_level_values(0).unique())[0]))
    ax.set_title(f"{store} — category mix by month (% of revenue)")
    ax.set_ylabel("Share")
    ax.set_xlabel("Month")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
fig.suptitle("Monthly category mix (100% stacked)")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_category_mix_month.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, store in zip(axes, sorted(order_agg["store_location"].unique()), strict=True):
    sub = order_agg[order_agg["store_location"] == store]
    ax.hist(sub["n_lines"], bins=range(1, int(sub["n_lines"].max()) + 2), alpha=0.7, color=store_palette[store])
    ax.set_title(f"{store} — lines per order")
    ax.set_xlabel("Number of line items")
    ax.set_ylabel("Orders")
fig.suptitle("Basket size (line items per order)")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_basket_size_lines.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, store in zip(axes, sorted(order_agg["store_location"].unique()), strict=True):
    sub = order_agg[order_agg["store_location"] == store]
    s = sub["order_revenue"].dropna()
    ax.boxplot(s.to_numpy(), vert=True, showfliers=True)
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    if iqr > 0:
        lo = max(0.0, q1 - 2.5 * iqr)
        hi = q3 + 2.5 * iqr
    else:
        lo, hi = float(s.quantile(0.01)), float(s.quantile(0.99))
    span = hi - lo
    pad = max(span * 0.1, 0.5) if span > 0 else 0.5
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_title(store)
    ax.set_ylabel("Order revenue")
fig.suptitle(
    "Order revenue distribution (y-axis zoomed to bulk of orders; extreme fliers may be clipped)"
)
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_order_revenue_box.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, store in zip(axes, sorted(order_agg["store_location"].unique()), strict=True):
    sub = order_agg[order_agg["store_location"] == store]
    s = sub["order_revenue"].dropna()
    ax.boxplot(s.to_numpy(), vert=True, showfliers=True)
    ax.set_title(store)
    ax.set_ylabel("Order revenue")
fig.suptitle("Order revenue distribution (full scale, all observations)")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_order_revenue_box_full_scale.png", bbox_inches="tight")
plt.show()

In [ ]:
order_agg["dow"] = pd.to_datetime(order_agg["date"]).dt.dayofweek
aov_dow = order_agg.groupby(["store_location", "dow"], observed=True)["order_revenue"].agg(
    mean_aov="mean", median_aov="median"
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
for ax, store in zip(axes, sorted(aov_dow["store_location"].unique()), strict=True):
    sub = aov_dow[aov_dow["store_location"] == store]
    x = sub["dow"]
    ax.plot(x, sub["mean_aov"], marker="o", label="Mean AOV", color=store_palette[store])
    ax.plot(x, sub["median_aov"], marker="s", linestyle="--", label="Median AOV", color="0.3")
    ax.set_xticks(range(7))
    ax.set_xticklabels(dow_labels)
    ax.set_title(store)
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Order revenue")
    ax.legend(fontsize=8)
fig.suptitle("AOV by day of week")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_aov_by_dow.png", bbox_inches="tight")
plt.show()

df_orders = df.drop_duplicates(subset=["order_id"])[["order_id", "store_location", "hour"]].merge(
    order_agg[["order_id", "order_revenue"]], on="order_id", how="inner"
)
aov_hour = df_orders.groupby(["store_location", "hour"], observed=True)["order_revenue"].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, store in zip(axes, sorted(aov_hour["store_location"].unique()), strict=True):
    sub = aov_hour[aov_hour["store_location"] == store].sort_values("hour")
    ax.bar(sub["hour"], sub["order_revenue"], color=store_palette[store], width=0.8)
    ax.set_title(store)
    ax.set_xlabel("Hour")
    ax.set_ylabel("Mean order revenue")
fig.suptitle("Mean AOV by hour of sale")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_aov_by_hour.png", bbox_inches="tight")
plt.show()

---

## Section 7 — Aggregation and correlation (person 2)

- **Daily demand (units)** per `product_detail` and per `product_id`, split by `store_location`; exported for forecasting and stocking (CSV under `outputs/`).
- **Orders vs line items:** for each store × day, compare distinct `order_id` counts to row counts, report **lines per order** and Pearson correlation (nearly collinear at daily level, but slope and residual spread matter).
- **Correlation heatmap** over numeric and calendar fields (and store id) at **line level** to highlight multicollinearity before regression or ML; strongest pairs are tabulated.


In [ ]:
# --- 7.1 Daily unit demand by store × SKU ---
daily_demand_detail = (
    df.groupby(["date", "store_location", "product_detail"], observed=True)["transaction_qty"]
    .sum()
    .reset_index(name="daily_qty")
)
daily_demand_pid = (
    df.groupby(["date", "store_location", "product_id"], observed=True)["transaction_qty"]
    .sum()
    .reset_index(name="daily_qty")
)

demand_detail_path = OUTPUTS_DIR / "daily_demand_product_detail.csv"
demand_pid_path = OUTPUTS_DIR / "daily_demand_product_id.csv"
daily_demand_detail.to_csv(demand_detail_path, index=False)
daily_demand_pid.to_csv(demand_pid_path, index=False)
print("Wrote:", demand_detail_path, "rows:", f"{len(daily_demand_detail):,}")
print("Wrote:", demand_pid_path, "rows:", f"{len(daily_demand_pid):,}")

mean_daily_by_sku = (
    daily_demand_detail.groupby(["store_location", "product_detail"], observed=True)["daily_qty"]
    .mean()
    .reset_index(name="mean_daily_qty")
    .sort_values(["store_location", "mean_daily_qty"], ascending=[True, False])
)
print("Top mean daily units by store (first 8 SKUs per store):")
display(mean_daily_by_sku.groupby("store_location", observed=True).head(8))

N_TOP_DEMAND = 10
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
for ax, store in zip(axes, sorted(mean_daily_by_sku["store_location"].unique()), strict=True):
    sub = mean_daily_by_sku[mean_daily_by_sku["store_location"] == store].head(N_TOP_DEMAND).iloc[::-1]
    ax.barh(sub["product_detail"].astype(str), sub["mean_daily_qty"], color=store_palette[store])
    ax.set_title(f"{store}\nmean daily units (top {N_TOP_DEMAND})")
    ax.set_xlabel("Mean units sold per day")
fig.suptitle("Average daily unit demand by product_detail")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_mean_daily_demand_top_sku.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 7.2 Daily order count vs line count (per store × day) ---
daily_lines = (
    df.groupby(["date", "store_location"], observed=True).size().reset_index(name="n_lines")
)
daily_orders = (
    df.groupby(["date", "store_location"], observed=True)["order_id"]
    .nunique()
    .reset_index(name="n_orders")
)
daily_vol = daily_lines.merge(daily_orders, on=["date", "store_location"])
daily_vol["lines_per_order"] = daily_vol["n_lines"] / daily_vol["n_orders"]

print("Lines per order (daily ratio) — summary:")
display(daily_vol["lines_per_order"].describe())

for store in sorted(daily_vol["store_location"].unique()):
    sub = daily_vol[daily_vol["store_location"] == store]
    r = sub["n_orders"].corr(sub["n_lines"])
    print(f"{store}: Pearson r(n_orders, n_lines) = {r:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, store in zip(axes, sorted(daily_vol["store_location"].unique()), strict=True):
    sub = daily_vol[daily_vol["store_location"] == store]
    ax.scatter(sub["n_orders"], sub["n_lines"], alpha=0.35, s=14, color=store_palette[store])
    z = np.polyfit(sub["n_orders"].to_numpy(), sub["n_lines"].to_numpy(), 1)
    x_line = np.linspace(sub["n_orders"].min(), sub["n_orders"].max(), 50)
    ax.plot(x_line, np.poly1d(z)(x_line), color="0.2", lw=1.5)
    ax.set_title(store)
    ax.set_xlabel("Distinct orders")
    ax.set_ylabel("Line items")
fig.suptitle("Daily orders vs line items (by store)")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_orders_vs_lines_scatter.png", bbox_inches="tight")
plt.show()


In [ ]:
# --- 7.3 Correlation matrix: numeric and calendar features (line level) ---
corr_cols = [
    "transaction_qty",
    "unit_price",
    "line_revenue",
    "hour",
    "dow",
    "month",
    "is_weekend",
    "year",
    "week",
    "is_month_start",
    "is_month_end",
    "is_holiday",
    "days_to_next_holiday",
    "is_drink",
    "is_hot_drink",
    "store_id",
]
corr_df = df[corr_cols].copy()
nunique = corr_df.nunique()
const_cols = nunique[nunique <= 1].index.tolist()
if const_cols:
    print("Dropping constant columns for correlation:", const_cols)
    corr_df = corr_df.drop(columns=const_cols)

corr_mat = corr_df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(
    corr_mat,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_title("Correlation matrix (line-level features, lower triangle)")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "fig_correlation_features.png", bbox_inches="tight")
plt.show()

pairs = (
    corr_mat.where(np.tril(np.ones_like(corr_mat, dtype=bool), k=-1))
    .stack()
    .reset_index()
)
pairs.columns = ["feature_a", "feature_b", "r"]
pairs["abs_r"] = pairs["r"].abs()
print("Top 15 pairs by |r|:")
display(pairs.sort_values("abs_r", ascending=False).head(15))


---

## Section 8 — Forecasting

**Objective:** Build SARIMAX demand forecasting models at two levels of granularity:

1. **Weekly revenue per store** (3 series) — the primary target for inventory budgeting.
2. **Weekly units per store × product category (coarse)** (6 series: drinks vs food/merch × 3 stores) — feeds Section 9 order quantities.

**Approach** (aligned with `semestral_project` Sections 5–5.8):
- Drop the partial Jan-1 week **and** the partial Jun-26 week (5 days only); keep **25 complete ISO weeks** (Jan 9 – Jun 19 2023).
- Train on the first **20 weeks** (Jan 9 – May 15), test on the last **5 weeks** (May 22 – Jun 19).
- Fit `SARIMAX(1,1,1)(1,0,0)[4]` — same spec used for per-item series in Section 5.6 of `semestral_project`; seasonal period 4 ≈ one month.
- Forecast distribution: report **mean, model σ, empirical σ (residual std × √h), and 80 / 95% CI** (both model and empirical) for every test week.
- Evaluate with **RMSE, MAE, MAPE, and RMSE/σ ratio** on held-out test weeks.
- Export `d_forecast` and `d_std` based on **observed** test-period demand (not model prediction) for Section 9.


### 8.1  Imports and weekly aggregation


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

DRINK_CATEGORIES = {'Coffee','Tea','Drinking Chocolate','Loose Tea','Coffee beans','Flavours'}
FOOD_CATEGORIES  = {'Bakery','Packaged Chocolate','Branded'}

# ── coarse category tag on line-level df ─────────────────────────────────
df['cat_coarse'] = df['product_category'].apply(
    lambda x: 'drinks' if x in DRINK_CATEGORIES else 'food_merch'
)

# ── ISO week start (Monday) ───────────────────────────────────────────────
df['week_start'] = df['date'] - pd.to_timedelta(df['date'].dt.dayofweek, unit='D')

# ── Drop partial Jan-1 week (only one day: 2023-01-01 is a Sunday) ────────
FIRST_FULL_WEEK = pd.Timestamp('2023-01-02')   # Monday of the first complete week
df_full = df[df['date'] >= FIRST_FULL_WEEK].copy()
df_full['week_start'] = df_full['date'] - pd.to_timedelta(df_full['date'].dt.dayofweek, unit='D')

# ── Weekly revenue per store ──────────────────────────────────────────────
weekly_store_rev = (
    df_full.groupby(['week_start', 'store_location'], observed=True)['line_revenue']
    .sum().reset_index(name='weekly_revenue')
)

# ── Weekly units per store × coarse category ─────────────────────────────
weekly_store_cat = (
    df_full.groupby(['week_start', 'store_location', 'cat_coarse'], observed=True)['transaction_qty']
    .sum().reset_index(name='weekly_qty')
)

stores  = sorted(weekly_store_rev['store_location'].unique())
cats    = sorted(weekly_store_cat['cat_coarse'].unique())
weeks   = sorted(weekly_store_rev['week_start'].unique())
N_WEEKS = len(weeks)

print(f'Complete weeks      : {N_WEEKS}  ({weeks[0].date()} → {weeks[-1].date()})')
print(f'Stores              : {stores}')
print(f'Coarse categories   : {cats}')
print(f'Revenue series rows : {len(weekly_store_rev)}')
print(f'Cat-demand series rows: {len(weekly_store_cat)}')
weekly_store_rev.pivot(index='week_start', columns='store_location', values='weekly_revenue').describe().round(0)


### 8.2  Seasonal decomposition — weekly revenue

Additive decomposition with period = 4 weeks (approximately one month). We decompose each store's revenue series independently.


In [ ]:
fig, axes = plt.subplots(len(stores), 4, figsize=(16, 4 * len(stores)), sharex=False)

for row_i, store in enumerate(stores):
    ts = (
        weekly_store_rev[weekly_store_rev['store_location'] == store]
        .set_index('week_start')['weekly_revenue']
        .sort_index()
    )
    decomp = seasonal_decompose(ts, model='additive', period=4)
    components = [
        (ts,             'Observed'),
        (decomp.trend,   'Trend'),
        (decomp.seasonal,'Seasonal (p=4)'),
        (decomp.resid,   'Residual'),
    ]
    colours = ['steelblue','darkorange','seagreen','crimson']
    for col_i, ((series, label), colour) in enumerate(zip(components, colours)):
        ax = axes[row_i, col_i]
        ax.plot(series, color=colour, linewidth=1.4)
        ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
        if row_i == 0:
            ax.set_title(label, fontsize=9)
        if col_i == 0:
            ax.set_ylabel(store, fontsize=8)

fig.suptitle('Additive seasonal decomposition — weekly revenue by store (period = 4 weeks)',
             fontsize=11)
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / 'fig_s8_decomposition.png', bbox_inches='tight')
plt.show()


**Interpretation of the seasonal decomposition (weekly revenue)**

- **Astoria:** clear upward trend over time, strong repeating 4-week seasonal cycle, and moderate residual spikes (some weeks over/under-perform the expected pattern).
- **Hell's Kitchen:** the steepest growth trend, very strong seasonality with larger swings than Astoria, and the largest residual outliers, which suggests more volatile shocks.
- **Lower Manhattan:** steady upward trend similar to the other stores, clear and stable 4-week seasonality, and residuals that are present but generally less extreme than Hell's Kitchen.

Overall, all three stores show **growth + regular monthly-like seasonality**. The main difference is volatility: **Hell's Kitchen appears most volatile**, while Astoria and Lower Manhattan are comparatively more stable around their trend + seasonal pattern.

### 8.3  Train / test split

After dropping both partial weeks (Jan-1 Sunday and Jun-26 5-day week), we have **25 complete ISO weeks**. We train on the first **20 weeks** (Jan 9 – May 15) and hold out the last **5 weeks** (May 22 – Jun 19) as the test set. This mirrors the 80/20 train/test ratio used in `semestral_project` Section 5.3.


In [ ]:
# ── Drop partial last week (Jun 26 – Jun 30, only 5 of 7 days) ───────────
# The dataset ends on Jun 30 (Friday). The ISO week starting Jun 26 contains
# only Mon–Fri, so its revenue/qty is ~30% lower than a full week by construction.
# Comparing the model's full-week forecast against this partial observation
# inflates RMSE and MAPE artificially. We drop it and work with 25 complete weeks.
days_per_week = (
    df_full.groupby('week_start')['date'].nunique()
)
complete_weeks_mask = days_per_week == 7
partial = days_per_week[~complete_weeks_mask]
if not partial.empty:
    print('Partial weeks dropped:')
    for w, d in partial.items():
        print(f'  {w.date()}  →  {d} days')

weeks   = sorted(days_per_week[complete_weeks_mask].index)
N_WEEKS = len(weeks)

# Also filter the weekly aggregations to complete weeks only
weekly_store_rev = weekly_store_rev[weekly_store_rev['week_start'].isin(weeks)].copy()
weekly_store_cat = weekly_store_cat[weekly_store_cat['week_start'].isin(weeks)].copy()

N_TEST  = 5
N_TRAIN = N_WEEKS - N_TEST
CUTOFF  = weeks[N_TRAIN]   # first week of the test set

print(f'\nComplete weeks      : {N_WEEKS}  ({weeks[0].date()} → {weeks[-1].date()})')
print(f'Train : {N_TRAIN} weeks  ({weeks[0].date()} → {weeks[N_TRAIN - 1].date()})')
print(f'Test  : {N_TEST}  weeks  ({weeks[N_TRAIN].date()} → {weeks[-1].date()})')
print(f'Cut-off (first test week): {CUTOFF.date()}')


### 8.4  SARIMAX weekly revenue — per store

Model: **SARIMAX(1,1,1)(1,0,0)[4]** — same spec as per-item demand models in `semestral_project` Section 5.6.  With only **20 training weeks**, a seasonal-difference order (1,1,0)[4] would leave too few effective observations after double-differencing, so we use seasonal AR(1) without seasonal differencing.

Two uncertainty bands are reported:
- **Model CI** (green): derived from `var_pred_mean` — captures parameter uncertainty only; tends to be too narrow when the trend extrapolates beyond the test period.
- **Empirical CI** (red): derived from in-sample residual std scaled by √h — captures actual forecast error magnitude and correctly widens with horizon.


In [ ]:
rev_forecasts    = {}   # store → predicted_mean Series
rev_forecast_std = {}   # store → model forecast std Series
rev_ci95         = {}   # store → model 95% CI DataFrame
rev_ci80         = {}   # store → model 80% CI DataFrame
rev_emp_ci95     = {}   # store → empirical 95% CI (actual-error-based)
rev_emp_ci80     = {}   # store → empirical 80% CI
rev_metrics      = {}   # store → metric dict
rev_models       = {}   # store → fitted result

for store in stores:
    ts = (
        weekly_store_rev[weekly_store_rev['store_location'] == store]
        .set_index('week_start')['weekly_revenue']
        .sort_index()
    )
    tr = ts[ts.index < CUTOFF]
    te = ts[ts.index >= CUTOFF]

    m = SARIMAX(tr, order=(1,1,1), seasonal_order=(1,0,0,4),
                enforce_stationarity=False, enforce_invertibility=False)
    r = m.fit(disp=False)
    rev_models[store] = r

    fc   = r.get_forecast(steps=len(te))
    mean = fc.predicted_mean.clip(lower=0)
    # ── model-internal forecast std (captures parameter uncertainty only) ──
    std_model = np.sqrt(fc.var_pred_mean)
    ci95_model = fc.conf_int(alpha=0.05).clip(lower=0)
    ci80_model = fc.conf_int(alpha=0.20).clip(lower=0)

    # ── empirical CI: use in-sample one-step residual std as error scale ──
    # This captures the actual forecast error magnitude (which the model
    # underestimates when the trend extrapolates beyond the test period).
    insample_resid_std = float(r.resid.std())
    # Scale by sqrt(h) for h-step ahead (conservative widening)
    h = np.arange(1, len(te) + 1)
    emp_std = insample_resid_std * np.sqrt(h)
    emp_std_s = pd.Series(emp_std, index=mean.index)
    z95, z80 = 1.96, 1.28
    emp_ci95 = pd.DataFrame({
        'lower': (mean - z95 * emp_std_s).clip(lower=0),
        'upper':  mean + z95 * emp_std_s,
    })
    emp_ci80 = pd.DataFrame({
        'lower': (mean - z80 * emp_std_s).clip(lower=0),
        'upper':  mean + z80 * emp_std_s,
    })

    rev_forecasts[store]    = mean
    rev_forecast_std[store] = std_model
    rev_ci95[store]         = ci95_model
    rev_ci80[store]         = ci80_model
    rev_emp_ci95[store]     = emp_ci95
    rev_emp_ci80[store]     = emp_ci80

    rmse = np.sqrt(mean_squared_error(te, mean))
    mae  = mean_absolute_error(te, mean)
    mape = np.mean(np.abs((te.values - mean.values) / te.values.clip(1))) * 100
    rmse_sigma_ratio = rmse / std_model.mean()   # >2 → model CI too narrow
    rev_metrics[store] = {
        'RMSE': rmse, 'MAE': mae, 'MAPE (%)': mape,
        'Model Avg σ': std_model.mean(),
        'RMSE/σ ratio': rmse_sigma_ratio,
        'Mean actual': te.mean(),
    }

metrics_rev_df = pd.DataFrame(rev_metrics).T.round(2)
print('── Weekly revenue forecast accuracy (test period) ──')
print(metrics_rev_df.to_string())
print('\nNote: RMSE/σ >> 1 means model CIs are too narrow (overconfident);')
print('      empirical CIs (plotted as dashed) widen the bands accordingly.')


### 8.5  Revenue forecast plots — mean ± uncertainty bands


In [ ]:
fig, axes = plt.subplots(len(stores), 1, figsize=(13, 4 * len(stores)), sharex=False)

for ax, store in zip(axes, stores):
    ts = (
        weekly_store_rev[weekly_store_rev['store_location'] == store]
        .set_index('week_start')['weekly_revenue'].sort_index()
    )
    tr = ts[ts.index < CUTOFF]
    te = ts[ts.index >= CUTOFF]
    
    mean = rev_forecasts[store]
    ci95_m = rev_ci95[store]
    ci80_m = rev_ci80[store]
    ci95_e = rev_emp_ci95[store]
    ci80_e = rev_emp_ci80[store]

    ax.plot(tr.index, tr, color='steelblue', linewidth=1.5, label='Train (actual)')
    ax.plot(te.index, te, color='darkorange', linewidth=1.8, label='Test (actual)')
    ax.plot(mean.index, mean, color='seagreen', linewidth=2,
            linestyle='--', label='Forecast mean')

    # === FIXED: Handle both column formats (0/1 or 'lower'/'upper') ===
    def get_ci_bounds(ci_df):
        if 'lower' in ci_df.columns and 'upper' in ci_df.columns:
            return ci_df['lower'], ci_df['upper']
        else:
            # Most common case from statsmodels conf_int(): columns 0 and 1
            return ci_df.iloc[:, 0], ci_df.iloc[:, 1]

    # Model CI (narrow — parameter uncertainty only)
    lower95m, upper95m = get_ci_bounds(ci95_m)
    lower80m, upper80m = get_ci_bounds(ci80_m)
    ax.fill_between(ci95_m.index, lower95m, upper95m,
                    alpha=0.15, color='seagreen', label='Model 95% CI')
    ax.fill_between(ci80_m.index, lower80m, upper80m,
                    alpha=0.25, color='seagreen', label='Model 80% CI')

    # Empirical CI (wider — residual-based)
    lower95e, upper95e = get_ci_bounds(ci95_e)
    lower80e, upper80e = get_ci_bounds(ci80_e)
    ax.fill_between(ci95_e.index, lower95e, upper95e,
                    alpha=0.10, color='crimson', label='Empirical 95% CI')
    ax.fill_between(ci80_e.index, lower80e, upper80e,
                    alpha=0.20, color='crimson', label='Empirical 80% CI')

    ax.axvline(CUTOFF, color='grey', linestyle=':', linewidth=1, label='Cut-off')

    rmse = rev_metrics[store]['RMSE']
    mape = rev_metrics[store]['MAPE (%)']
    ratio = rev_metrics[store]['RMSE/σ ratio']
    
    ax.set_title(
        f'{store} — RMSE=${rmse:,.0f}  MAPE={mape:.1f}% '
        f'(RMSE/σ={ratio:.1f}× → model CI overconfident)',
        fontsize=9
    )
    ax.set_ylabel('Weekly revenue ($)')
    ax.legend(fontsize=7, ncol=3, loc='upper left')

fig.suptitle(
    'SARIMAX(1,1,1)(1,0,0)[4] — weekly revenue forecast\n'
    'Green bands: model CI (parameter uncertainty only) | '
    'Red bands: empirical CI (residual std × √h)',
    fontsize=11
)
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / 'fig_s8_revenue_forecast.png', bbox_inches='tight')
plt.show()

### 8.6  SARIMAX weekly units — per store × category

We fit one **SARIMAX(1,1,1)(1,0,0)[4]** per store × coarse-category combination (6 models total), each trained on **20 weeks**. Both model and empirical CI bands are computed as in Section 8.4.

The demand inputs for Section 9 (`d_forecast`, `d_std`) are derived from **observed** test-period daily demand — not from the model's predicted mean — because the model over-extrapolated the training trend (RMSE/σ = 3–7×).


In [ ]:
cat_forecasts     = {}   # (store, cat) → predicted_mean
cat_forecast_std  = {}   # (store, cat) → model forecast std
cat_ci95          = {}   # (store, cat) → model 95% CI
cat_ci80          = {}   # (store, cat) → model 80% CI
cat_emp_ci95      = {}   # (store, cat) → empirical 95% CI
cat_emp_ci80      = {}   # (store, cat) → empirical 80% CI
cat_metrics       = {}   # (store, cat) → metric dict

for store in stores:
    for cat in cats:
        key = (store, cat)
        ts = (
            weekly_store_cat[
                (weekly_store_cat['store_location'] == store) &
                (weekly_store_cat['cat_coarse'] == cat)
            ].set_index('week_start')['weekly_qty'].sort_index()
        )
        ts = ts.reindex(weeks, fill_value=0)
        tr = ts[ts.index < CUTOFF]
        te = ts[ts.index >= CUTOFF]

        m = SARIMAX(tr, order=(1,1,1), seasonal_order=(1,0,0,4),
                    enforce_stationarity=False, enforce_invertibility=False)
        r = m.fit(disp=False)

        fc        = r.get_forecast(steps=len(te))
        mean      = fc.predicted_mean.clip(lower=0)
        std_model = np.sqrt(fc.var_pred_mean)
        ci95_m    = fc.conf_int(alpha=0.05).clip(lower=0)
        ci80_m    = fc.conf_int(alpha=0.20).clip(lower=0)

        # ── empirical CI (in-sample residual std, √h scaling) ────────────
        insample_resid_std = float(r.resid.std())
        h = np.arange(1, len(te) + 1)
        emp_std = insample_resid_std * np.sqrt(h)
        emp_std_s = pd.Series(emp_std, index=mean.index)
        z95, z80 = 1.96, 1.28
        emp_ci95 = pd.DataFrame({
            'lower': (mean - z95 * emp_std_s).clip(lower=0),
            'upper':  mean + z95 * emp_std_s,
        })
        emp_ci80 = pd.DataFrame({
            'lower': (mean - z80 * emp_std_s).clip(lower=0),
            'upper':  mean + z80 * emp_std_s,
        })

        cat_forecasts[key]    = mean
        cat_forecast_std[key] = std_model
        cat_ci95[key]         = ci95_m
        cat_ci80[key]         = ci80_m
        cat_emp_ci95[key]     = emp_ci95
        cat_emp_ci80[key]     = emp_ci80

        rmse  = np.sqrt(mean_squared_error(te, mean))
        mae   = mean_absolute_error(te, mean)
        mape  = np.mean(np.abs((te.values - mean.values) / te.values.clip(1))) * 100
        ratio = rmse / std_model.mean()
        cat_metrics[key] = {
            'RMSE': round(rmse, 1), 'MAE': round(mae, 1),
            'MAPE (%)': round(mape, 1),
            'Model Avg σ': round(std_model.mean(), 1),
            'RMSE/σ': round(ratio, 1),
            'Mean actual': round(te.mean(), 1),
        }

metrics_cat_df = pd.DataFrame(cat_metrics).T
metrics_cat_df.index.names = ['store', 'category']
print('── Weekly unit demand forecast accuracy (test period) ──')
print(metrics_cat_df.to_string())
print('\nNote: RMSE/σ >> 1 means model CIs understate true forecast uncertainty.')


### 8.7  Per-store × category forecast plots


In [ ]:
fig, axes = plt.subplots(len(stores), len(cats),
                          figsize=(7 * len(cats), 4 * len(stores)), sharex=False)

for row_i, store in enumerate(stores):
    for col_i, cat in enumerate(cats):
        ax  = axes[row_i, col_i]
        key = (store, cat)
        ts  = (
            weekly_store_cat[
                (weekly_store_cat['store_location'] == store) &
                (weekly_store_cat['cat_coarse'] == cat)
            ].set_index('week_start')['weekly_qty'].sort_index().reindex(weeks, fill_value=0)
        )
        tr     = ts[ts.index < CUTOFF]
        te     = ts[ts.index >= CUTOFF]
        mean   = cat_forecasts[key]
        ci95_m = cat_ci95[key]
        ci80_m = cat_ci80[key]
        ci95_e = cat_emp_ci95[key]
        ci80_e = cat_emp_ci80[key]

        ax.plot(tr.index, tr, color='steelblue', linewidth=1.2, label='Train')
        ax.plot(te.index, te, color='darkorange', linewidth=1.5, label='Actual')
        ax.plot(mean.index, mean, color='seagreen', linewidth=2,
                linestyle='--', label='Forecast mean')
        ax.fill_between(ci95_m.index, ci95_m.iloc[:,0], ci95_m.iloc[:,1],
                        alpha=0.15, color='seagreen', label='Model 95% CI')
        ax.fill_between(ci80_m.index, ci80_m.iloc[:,0], ci80_m.iloc[:,1],
                        alpha=0.25, color='seagreen', label='Model 80% CI')
        ax.fill_between(ci95_e.index, ci95_e['lower'], ci95_e['upper'],
                        alpha=0.10, color='crimson', label='Empirical 95% CI')
        ax.fill_between(ci80_e.index, ci80_e['lower'], ci80_e['upper'],
                        alpha=0.20, color='crimson', label='Empirical 80% CI')
        ax.axvline(CUTOFF, color='grey', linestyle=':', linewidth=1)

        ratio = cat_metrics[key]['RMSE/σ']
        rmse  = cat_metrics[key]['RMSE']
        ax.set_title(
            f'{store}\n{cat}  RMSE={rmse:.0f}  RMSE/σ={ratio:.1f}×',
            fontsize=8
        )
        ax.set_ylabel('Weekly units')
        if row_i == 0 and col_i == 0:
            ax.legend(fontsize=6, loc='upper left')

fig.suptitle(
    'SARIMAX(1,1,1)(1,0,0)[4] — weekly unit demand by store × category\n'
    'Green: model CI  |  Red: empirical CI (residual std × √h)',
    fontsize=11, y=1.01
)
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / 'fig_s8_cat_forecasts.png', bbox_inches='tight')
plt.show()


### 8.8  Daily demand summary — export for Section 9

Rather than converting the model's (over-extrapolated) predicted weekly mean to daily figures, we use the **observed actual daily demand** from the 5 held-out test weeks (May 22 – Jun 19). This is the most recent stable demand window and avoids the bias introduced by the upward training trend.

`daily_std` is the empirical standard deviation of actual daily demand in the test window — the correct input for safety-stock sizing in Section 9. The model's own predicted mean is retained as `model_daily_mean` for reference.


In [ ]:
# ── daily_mean and daily_std: observed values from the held-out test period ──
# Problem with using predicted_mean.mean()/7: the model over-extrapolated the
# upward trend, producing daily_mean ~50% above actual demand. The model's
# var_pred_mean also understates real uncertainty (RMSE/σ = 3–7×).
#
# Fix: use the actual observed daily demand during the test weeks as the demand
# estimate for Section 9. The test period (May 22 – Jun 19) is the most recent
# 5-week window of stable, plateau-level demand — more representative of the
# near-future operating environment than the training ramp-up period.
#
# daily_std is the empirical std of actual daily demand in the test window,
# which correctly captures operational variability for safety-stock sizing.

test_dates = df_full[
    (df_full['week_start'].isin(weeks)) &
    (df_full['week_start'] >= CUTOFF)
]['date'].unique()

demand_rows = []
for store in stores:
    for cat in cats:
        key = (store, cat)

        # ── observed daily qty in test period ────────────────────────────
        test_daily = (
            df_full[
                (df_full['store_location'] == store) &
                (df_full['cat_coarse']     == cat)   &
                (df_full['date'].isin(test_dates))
            ].groupby('date')['transaction_qty'].sum()
        )
        daily_mean_obs = test_daily.mean()
        daily_std_obs  = test_daily.std()
        cv_pct         = (daily_std_obs / daily_mean_obs * 100) if daily_mean_obs > 0 else np.nan

        # ── weekday / weekend split from test period ──────────────────────
        wd_mean = test_daily[test_daily.index.dayofweek < 5].mean()
        we_mean = test_daily[test_daily.index.dayofweek >= 5].mean()

        # ── model forecast for reference (what SARIMAX predicted) ─────────
        model_daily_mean = cat_forecasts[key].mean() / 7

        demand_rows.append({
            'store'             : store,
            'category'          : cat,
            'daily_mean'        : round(daily_mean_obs,   2),
            'daily_std'         : round(daily_std_obs,    2),
            'cv_pct'            : round(cv_pct,           1),
            'test_weekday_mean' : round(wd_mean,          2),
            'test_weekend_mean' : round(we_mean,          2),
            'model_daily_mean'  : round(model_daily_mean, 2),
        })

demand_s9 = pd.DataFrame(demand_rows).set_index(['store','category'])
print('Daily demand summary for Section 9 — OBSERVED test-period values:')
print(demand_s9.to_string())
print()
print('Note: model_daily_mean (for reference) = SARIMAX predicted_mean / 7.')
print('      It exceeds daily_mean because the model over-extrapolated the upward trend.')
print('      Section 9 uses daily_mean (observed) and daily_std (observed) as d_i and σ_i.')

# ── scalar exports for Section 9 ─────────────────────────────────────────
d_forecast = demand_s9['daily_mean']        # observed test-period mean
d_std      = demand_s9['daily_std']         # observed test-period std
d_weekday  = demand_s9['test_weekday_mean']
d_weekend  = demand_s9['test_weekend_mean']

demand_s9.to_csv(OUTPUTS_DIR / 'demand_forecast_s9.csv')
print('\nd_forecast, d_std, d_weekday, d_weekend exported for Section 9.')
print('Saved: demand_forecast_s9.csv')


---

## Section 9 — Feature engineering & data preparation

**Objective:** Load the enriched transaction CSV and aggregate into a **daily demand** table per store per sized product type.

- **Data source:** `Coffee Shop Sales (1)(Transactions).csv` — pre-enriched with `product_type_sized` and `COGS_product_type` columns.
- `product_type_sized` distinguishes product types by their size variant (sm / rg / lg suffix).
- `COGS_product_type` carries the product-specific cost-of-goods-sold percentage (20 %, 30 %, or 40 %) — used in Section 12 for profit simulation.
- The result is a daily demand table: `date`, `store_location`, `product_type_sized`, `units_sold`, `avg_unit_price`, `cogs_pct`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

CSV_PATH = Path('.') / 'Coffee Shop Sales (1)(Transactions).csv'
txn = pd.read_csv(CSV_PATH, parse_dates=['transaction_date'])
txn.rename(columns={'transaction_date': 'date'}, inplace=True)

txn['cogs_pct'] = (
    txn['COGS_product_type']
    .str.rstrip('%')
    .astype(float) / 100.0
)

print(f"Loaded {len(txn):,} transactions from CSV")
print(f"Columns: {list(txn.columns)}")
print(f"\nUnique product_type values        : {txn['product_type'].nunique()}")
print(f"Unique product_type_sized values  : {txn['product_type_sized'].nunique()}")

print("\nSample mapping (first 15 distinct):")
sample = (
    txn[['product_type', 'product_detail', 'product_type_sized']]
    .drop_duplicates(subset='product_type_sized')
    .sort_values('product_type_sized')
    .head(15)
)
display(sample)

print("\nCOGS breakdown:")
display(txn.groupby('cogs_pct')['product_type'].nunique().reset_index(name='n_product_types'))

daily_demand = (
    txn.groupby(['date', 'store_location', 'product_type_sized'], observed=True)
    .agg(
        units_sold=('transaction_qty', 'sum'),
        avg_unit_price=('unit_price', 'mean'),
        cogs_pct=('cogs_pct', 'first'),
    )
    .reset_index()
    .sort_values(['store_location', 'product_type_sized', 'date'])
    .reset_index(drop=True)
)

print(f"\nDaily demand table shape: {daily_demand.shape}")
display(daily_demand.head(10))
print("\nStores:", sorted(daily_demand['store_location'].unique()))
print("Date range:", daily_demand['date'].min().date(), "→", daily_demand['date'].max().date())


---

## Section 10 — Baseline model (recency-weighted average)

**Objective:** Build a weighted-average baseline forecast for daily units sold per store per sized product type.

- Weights decay exponentially with distance from the prediction date: $w_i = e^{-\lambda \cdot \text{days\_ago}}$.
- $\lambda$ is chosen so that the most recent 14 days carry approximately 80 % of the total weight.
- The baseline serves as the benchmark against which all other models are compared.

**Train / test split (shared by all models):**
- **Training set:** all days except the last 6 weeks (42 days).
- **Test set:** the final 42 days.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

TEST_WEEKS = 6
TEST_DAYS  = TEST_WEEKS * 7

all_dates   = sorted(daily_demand['date'].unique())
cutoff_date = all_dates[-TEST_DAYS]

train = daily_demand[daily_demand['date'] <  cutoff_date].copy()
test  = daily_demand[daily_demand['date'] >= cutoff_date].copy()

print(f"Train: {train['date'].min().date()} → {train['date'].max().date()}  ({train['date'].nunique()} days)")
print(f"Test : {test['date'].min().date()} → {test['date'].max().date()}  ({test['date'].nunique()} days)")

LAMBDA = -np.log(0.20) / 14

def weighted_avg_forecast(group_train, target_date):
    if group_train.empty:
        return 0.0
    days_ago = (target_date - group_train['date']).dt.days.values
    weights  = np.exp(-LAMBDA * days_ago)
    return np.average(group_train['units_sold'].values, weights=weights)

baseline_preds = []
groups = train.groupby(['store_location', 'product_type_sized'])

for _, row in test.iterrows():
    key = (row['store_location'], row['product_type_sized'])
    if key in groups.groups:
        grp = groups.get_group(key)
        pred = weighted_avg_forecast(grp, row['date'])
    else:
        pred = 0.0
    baseline_preds.append(pred)

test = test.copy()
test['baseline_pred'] = baseline_preds

baseline_mae  = mean_absolute_error(test['units_sold'], test['baseline_pred'])
baseline_rmse = np.sqrt(mean_squared_error(test['units_sold'], test['baseline_pred']))

print(f"\nBaseline MAE  : {baseline_mae:.2f}")
print(f"Baseline RMSE : {baseline_rmse:.2f}")

for store in sorted(test['store_location'].unique()):
    mask = test['store_location'] == store
    mae  = mean_absolute_error(test.loc[mask, 'units_sold'], test.loc[mask, 'baseline_pred'])
    rmse = np.sqrt(mean_squared_error(test.loc[mask, 'units_sold'], test.loc[mask, 'baseline_pred']))
    print(f"  {store:20s}  MAE={mae:.2f}  RMSE={rmse:.2f}")


---

## Section 11 — Model development

**Models** (all fitted per store × sized product type):

1. **Simple moving average** (7-day and 14-day windows)
2. **Exponential smoothing** (Holt–Winters, weekly seasonality with period = 7)
3. **Linear regression** with features: day of week, week number, is_weekend, lagged demand (1 d, 7 d), rolling mean (7 d, 14 d)
4. **Random Forest regressor** with the same feature set

Each model predicts daily units for the 6-week test window. Results are collected into a single comparison DataFrame.


In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

def moving_avg_forecast(series_train, series_test_dates, window):
    preds = []
    combined = series_train.copy()
    for d in series_test_dates:
        tail = combined.tail(window)
        preds.append(tail.mean() if len(tail) > 0 else 0.0)
        actual_val = test.loc[
            (test['date'] == d) &
            (test['store_location'] == current_store) &
            (test['product_type_sized'] == current_pt),
            'units_sold'
        ]
        combined = pd.concat([combined, actual_val])
    return preds

def ets_forecast(series_train, n_test):
    try:
        if len(series_train) < 14:
            return [series_train.mean()] * n_test
        model = ExponentialSmoothing(
            series_train.values,
            trend='add',
            seasonal='add',
            seasonal_periods=7,
        ).fit(optimized=True)
        return model.forecast(n_test).tolist()
    except Exception:
        return [series_train.mean()] * n_test

def build_features(frame):
    f = frame.copy()
    f['dow']        = f['date'].dt.dayofweek
    week_iso = f['date'].dt.isocalendar().week
    f['week_num']   = week_iso.fillna(0).astype(int)
    f['is_weekend'] = (f['dow'] >= 5).astype(int)
    f['lag_1']      = f['units_sold'].shift(1)
    f['lag_7']      = f['units_sold'].shift(7)
    f['roll_7']     = f['units_sold'].rolling(7, min_periods=1).mean()
    f['roll_14']    = f['units_sold'].rolling(14, min_periods=1).mean()
    return f

FEATURE_COLS = ['dow', 'week_num', 'is_weekend', 'lag_1', 'lag_7', 'roll_7', 'roll_14']

stores_list = sorted(daily_demand['store_location'].unique())
pt_list     = sorted(daily_demand['product_type_sized'].unique())

all_results = []
n_combos = len(stores_list) * len(pt_list)
progress = 0

for current_store in stores_list:
    for current_pt in pt_list:
        progress += 1
        if progress % 50 == 0:
            print(f"  Processing {progress}/{n_combos} ...")

        sub = daily_demand[
            (daily_demand['store_location'] == current_store) &
            (daily_demand['product_type_sized'] == current_pt)
        ].sort_values('date').reset_index(drop=True)

        sub_train = sub[sub['date'] < cutoff_date].copy()
        sub_test  = sub[sub['date'] >= cutoff_date].copy()

        if sub_test.empty:
            continue

        test_dates = sub_test['date'].values
        n_test     = len(sub_test)

        ma7  = moving_avg_forecast(sub_train['units_sold'], test_dates, 7)
        ma14 = moving_avg_forecast(sub_train['units_sold'], test_dates, 14)
        ets  = ets_forecast(sub_train['units_sold'], n_test)

        full = build_features(sub)
        f_train = full[full['date'] < cutoff_date].dropna(subset=FEATURE_COLS)
        f_test  = full[full['date'] >= cutoff_date].copy()

        for col in FEATURE_COLS:
            f_test[col] = f_test[col].fillna(f_train[col].median() if not f_train.empty else 0)

        if len(f_train) >= 5 and not f_test.empty:
            X_tr = f_train[FEATURE_COLS].values
            y_tr = f_train['units_sold'].values
            X_te = f_test[FEATURE_COLS].values

            lr = LinearRegression().fit(X_tr, y_tr)
            lr_preds = np.maximum(lr.predict(X_te), 0)

            rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
            rf.fit(X_tr, y_tr)
            rf_preds = np.maximum(rf.predict(X_te), 0)
        else:
            fallback = sub_train['units_sold'].mean() if not sub_train.empty else 0
            lr_preds = [fallback] * n_test
            rf_preds = [fallback] * n_test

        baseline_vals = test.loc[
            (test['store_location'] == current_store) &
            (test['product_type_sized'] == current_pt),
            'baseline_pred'
        ].values

        if len(baseline_vals) != n_test:
            baseline_vals = np.full(n_test, sub_train['units_sold'].mean() if not sub_train.empty else 0)

        for i in range(n_test):
            all_results.append({
                'date':              sub_test.iloc[i]['date'],
                'store_location':    current_store,
                'product_type_sized': current_pt,
                'actual':            sub_test.iloc[i]['units_sold'],
                'avg_unit_price':    sub_test.iloc[i]['avg_unit_price'],
                'baseline_pred':     baseline_vals[i] if i < len(baseline_vals) else 0,
                'ma7_pred':          ma7[i],
                'ma14_pred':         ma14[i],
                'ets_pred':          ets[i] if i < len(ets) else 0,
                'lr_pred':           lr_preds[i] if i < len(lr_preds) else 0,
                'rf_pred':           rf_preds[i] if i < len(rf_preds) else 0,
            })

results_df = pd.DataFrame(all_results)
MODEL_COLS = ['baseline_pred', 'ma7_pred', 'ma14_pred', 'ets_pred', 'lr_pred', 'rf_pred']

print(f"\nResults shape: {results_df.shape}")
print(f"Models: {MODEL_COLS}")
print("\nOverall accuracy (test set):")
for col in MODEL_COLS:
    mae  = mean_absolute_error(results_df['actual'], results_df[col])
    rmse = np.sqrt(mean_squared_error(results_df['actual'], results_df[col]))
    print(f"  {col:18s}  MAE={mae:.2f}  RMSE={rmse:.2f}")

print("\nPer-store accuracy:")
for store in stores_list:
    mask = results_df['store_location'] == store
    print(f"\n  {store}:")
    for col in MODEL_COLS:
        mae  = mean_absolute_error(results_df.loc[mask, 'actual'], results_df.loc[mask, col])
        rmse = np.sqrt(mean_squared_error(results_df.loc[mask, 'actual'], results_df.loc[mask, col]))
        print(f"    {col:18s}  MAE={mae:.2f}  RMSE={rmse:.2f}")


---

## Section 12 — Risk and inventory strategy (standard deviation experiment)

**Business constraint:** unsold products are lost at the product-specific COGS rate (20 %, 30 %, or 40 % of selling price, depending on `COGS_product_type` from the CSV).

For each model's predictions we compute the **residual standard deviation** (σ) per (store, product type) and simulate stocking at different risk levels:

| Risk level | z-score | Interpretation |
|:-----------|:--------|:---------------|
| Conservative | 0.0 | Stock = forecast (higher stockout risk, lower waste) |
| Moderate | 0.5 | Modest safety buffer |
| Aggressive | 1.0 | Lower stockout risk, higher potential waste |
| Very aggressive | 1.5 | Maximum safety stock |

Daily simulation per product type per store (using per-product COGS):
- `sold = min(stock, actual_demand)`
- `unsold = max(0, stock − actual_demand)`
- `revenue = sold × unit_price`
- `waste_cost = unsold × unit_price × cogs_pct`
- `profit = revenue − stock × unit_price × cogs_pct`
- `stockout_qty = max(0, actual_demand − stock)`


In [ ]:
Z_SCORES = [0.0, 0.5, 1.0, 1.5]

cogs_lookup = (
    daily_demand
    .groupby(['store_location', 'product_type_sized'])['cogs_pct']
    .first()
)

residual_std = {}
for col in MODEL_COLS:
    residuals = results_df.groupby(['store_location', 'product_type_sized']).apply(
        lambda g: (g['actual'] - g[col]).std()
    )
    residual_std[col] = residuals

inventory_results = []

for _, row in results_df.iterrows():
    store  = row['store_location']
    pt     = row['product_type_sized']
    actual = row['actual']
    price  = row['avg_unit_price']
    cogs_pct = cogs_lookup.get((store, pt), 0.40)
    cogs   = price * cogs_pct

    for col in MODEL_COLS:
        pred  = max(row[col], 0)
        sigma = residual_std[col].get((store, pt), 0)
        if pd.isna(sigma):
            sigma = 0

        for z in Z_SCORES:
            stock       = max(pred + z * sigma, 0)
            sold        = min(stock, actual)
            unsold      = max(0, stock - actual)
            revenue     = sold * price
            waste_cost  = unsold * cogs
            total_cost  = stock * cogs
            profit      = revenue - total_cost
            stockout    = max(0, actual - stock)

            inventory_results.append({
                'date':              row['date'],
                'store_location':    store,
                'product_type_sized': pt,
                'model':             col.replace('_pred', ''),
                'z_score':           z,
                'actual':            actual,
                'predicted':         pred,
                'stock':             stock,
                'sold':              sold,
                'unsold':            unsold,
                'revenue':           revenue,
                'waste_cost':        waste_cost,
                'profit':            profit,
                'stockout_qty':      stockout,
                'cogs_pct':          cogs_pct,
            })

inv_df = pd.DataFrame(inventory_results)
print(f"Inventory simulation shape: {inv_df.shape}")

print("\nAggregate business metrics (across all stores, all product types):")
summary = (
    inv_df.groupby(['model', 'z_score'])
    .agg(
        total_revenue   = ('revenue', 'sum'),
        total_waste_cost= ('waste_cost', 'sum'),
        total_profit    = ('profit', 'sum'),
        total_stockout  = ('stockout_qty', 'sum'),
    )
    .reset_index()
)
display(summary)

print("\nPer-store summary at z=1.0:")
store_summary = (
    inv_df[inv_df['z_score'] == 1.0]
    .groupby(['store_location', 'model'])
    .agg(
        total_profit   = ('profit', 'sum'),
        total_waste    = ('waste_cost', 'sum'),
        total_stockout = ('stockout_qty', 'sum'),
    )
    .reset_index()
)
display(store_summary.sort_values(['store_location', 'total_profit'], ascending=[True, False]))


---

## Section 13 — Evaluation and comparison

Compare all models against the baseline and each other using both **forecast accuracy** metrics (MAE, RMSE, MAPE) and **business impact** metrics (profit, waste, stockouts) at each risk level.


### 11. Ad hoc tests
Use this section to quickly test one-off questions on the prepared `df` dataset.

## Section 14 - Pilot forecast (one model, one location, 7 test days)

This pilot keeps the scope intentionally small:

- one location
- one model (linear regression)
- first 7 days of the 6-week test window
- all available `product_type_sized` values for that location

Outputs:

1. day x product table with `baseline_pred`, `model_pred`, `actual`, and profit/waste comparison
2. product summary table over the 7 days with `profit_diff` (model minus baseline)

Assumptions for inventory economics:

- prepared stock = `ceil(prediction)`
- if prepared and unsold, it is thrown away
- unit cost is `avg_unit_price * cogs_pct`
- profit = `sold * price - prepared * unit_cost`
- waste cost = `unsold * unit_cost`

In [ ]:
from sklearn.linear_model import LinearRegression

TARGET_STORE = "Astoria"  # change if you want another location
BASELINE_LAMBDA = -np.log(0.20) / 14

all_dates = sorted(daily_demand["date"].dropna().unique())
if len(all_dates) == 0:
    raise ValueError("No dates found in daily_demand")

# Full 6-week test window
pilot_test_dates = [d for d in all_dates if d >= cutoff_date]
if len(pilot_test_dates) == 0:
    raise ValueError("No test dates found from cutoff_date onward")

pilot_products = sorted(
    daily_demand.loc[
        daily_demand["store_location"] == TARGET_STORE,
        "product_type_sized",
    ].dropna().unique()
)
if len(pilot_products) == 0:
    raise ValueError(f"No product_type_sized found for store {TARGET_STORE}")

meta = (
    daily_demand[daily_demand["store_location"] == TARGET_STORE]
    .groupby("product_type_sized", as_index=False)
    .agg(avg_unit_price=("avg_unit_price", "mean"), cogs_pct=("cogs_pct", "first"))
)

pilot_grid = pd.MultiIndex.from_product(
    [all_dates, pilot_products], names=["date", "product_type_sized"]
).to_frame(index=False)

store_obs = daily_demand[daily_demand["store_location"] == TARGET_STORE][
    ["date", "product_type_sized", "units_sold"]
]

pilot_grid = (
    pilot_grid
    .merge(store_obs, on=["date", "product_type_sized"], how="left")
    .merge(meta, on="product_type_sized", how="left")
)
pilot_grid["units_sold"] = pilot_grid["units_sold"].fillna(0)

pilot_grid = pilot_grid.sort_values(["product_type_sized", "date"]).reset_index(drop=True)
pilot_grid["dow"] = pilot_grid["date"].dt.dayofweek
pilot_grid["week_num"] = pilot_grid["date"].dt.isocalendar().week.fillna(0).astype(int)
pilot_grid["lag_1"] = pilot_grid.groupby("product_type_sized")["units_sold"].shift(1)
pilot_grid["lag_7"] = pilot_grid.groupby("product_type_sized")["units_sold"].shift(7)
pilot_grid["roll_7"] = (
    pilot_grid.groupby("product_type_sized")["units_sold"]
    .transform(lambda s: s.rolling(7, min_periods=1).mean())
)

num_cols = ["dow", "week_num", "lag_1", "lag_7", "roll_7"]
cat_cols = pd.get_dummies(pilot_grid["product_type_sized"], prefix="pt")
X_all = pd.concat([pilot_grid[num_cols], cat_cols], axis=1)

y_all = pilot_grid["units_sold"]
train_mask = pilot_grid["date"] < cutoff_date
pilot_mask = pilot_grid["date"].isin(pilot_test_dates)
fit_mask = train_mask & pilot_grid[num_cols].notna().all(axis=1)

model = LinearRegression()
model.fit(X_all.loc[fit_mask], y_all.loc[fit_mask])
pilot_grid["model_pred"] = np.nan
pilot_grid.loc[pilot_mask, "model_pred"] = np.maximum(model.predict(X_all.loc[pilot_mask]), 0)

train_hist = pilot_grid[train_mask][["date", "product_type_sized", "units_sold"]]


def weighted_avg_for_product(hist_df: pd.DataFrame, target_date: pd.Timestamp) -> float:
    if hist_df.empty:
        return 0.0
    days_ago = (target_date - hist_df["date"]).dt.days.values
    weights = np.exp(-BASELINE_LAMBDA * days_ago)
    return float(np.average(hist_df["units_sold"].values, weights=weights))


baseline_preds = []
for _, row in pilot_grid.loc[pilot_mask, ["date", "product_type_sized"]].iterrows():
    h = train_hist[train_hist["product_type_sized"] == row["product_type_sized"]]
    baseline_preds.append(weighted_avg_for_product(h, row["date"]))

pilot_grid.loc[pilot_mask, "baseline_pred"] = baseline_preds

pilot_out = pilot_grid.loc[pilot_mask, [
    "date", "product_type_sized", "units_sold", "avg_unit_price", "cogs_pct", "baseline_pred", "model_pred"
]].copy()

pilot_out = pilot_out.rename(columns={"units_sold": "actual"})
pilot_out["margin_rate"] = 1 - pilot_out["cogs_pct"]
pilot_out["unit_cost"] = pilot_out["avg_unit_price"] * pilot_out["cogs_pct"]

for prefix in ["baseline", "model"]:
    pred_col = f"{prefix}_pred"
    stock_col = f"{prefix}_stock"
    sold_col = f"{prefix}_sold"
    unsold_col = f"{prefix}_unsold"
    waste_col = f"{prefix}_waste_cost"
    profit_col = f"{prefix}_profit"

    pilot_out[stock_col] = np.ceil(np.maximum(pilot_out[pred_col], 0)).astype(int)
    pilot_out[sold_col] = np.minimum(pilot_out[stock_col], pilot_out["actual"])
    pilot_out[unsold_col] = np.maximum(pilot_out[stock_col] - pilot_out["actual"], 0)
    pilot_out[waste_col] = pilot_out[unsold_col] * pilot_out["unit_cost"]
    pilot_out[profit_col] = (pilot_out[sold_col] * pilot_out["avg_unit_price"]) - (pilot_out[stock_col] * pilot_out["unit_cost"])

pilot_out["profit_diff"] = pilot_out["model_profit"] - pilot_out["baseline_profit"]

pilot_day_product = pilot_out[[
    "date",
    "product_type_sized",
    "actual",
    "baseline_pred",
    "model_pred",
    "baseline_stock",
    "model_stock",
    "baseline_waste_cost",
    "model_waste_cost",
    "baseline_profit",
    "model_profit",
    "profit_diff",
    "margin_rate",
]].sort_values(["date", "product_type_sized"]).reset_index(drop=True)

pilot_day_product["week_start"] = pilot_day_product["date"] - pd.to_timedelta(
    pilot_day_product["date"].dt.dayofweek, unit="D"
)

weekly_profit_compare = (
    pilot_day_product.groupby("week_start", as_index=False)
    .agg(
        baseline_profit=("baseline_profit", "sum"),
        model_profit=("model_profit", "sum"),
        baseline_waste=("baseline_waste_cost", "sum"),
        model_waste=("model_waste_cost", "sum"),
        baseline_pred=("baseline_pred", "sum"),
        model_pred=("model_pred", "sum"),
        actual=("actual", "sum"),
    )
)
weekly_profit_compare["profit_diff"] = (
    weekly_profit_compare["model_profit"] - weekly_profit_compare["baseline_profit"]
)
weekly_profit_compare = weekly_profit_compare.sort_values("week_start").reset_index(drop=True)

overall_profit_compare = pd.DataFrame([
    {
        "period": "full_6_weeks",
        "baseline_profit": pilot_day_product["baseline_profit"].sum(),
        "model_profit": pilot_day_product["model_profit"].sum(),
        "baseline_waste": pilot_day_product["baseline_waste_cost"].sum(),
        "model_waste": pilot_day_product["model_waste_cost"].sum(),
        "baseline_pred": pilot_day_product["baseline_pred"].sum(),
        "model_pred": pilot_day_product["model_pred"].sum(),
        "actual": pilot_day_product["actual"].sum(),
        "profit_diff": pilot_day_product["profit_diff"].sum(),
    }
])

print(f"Target store: {TARGET_STORE}")
print(f"Test window: {pilot_test_dates[0].date()} to {pilot_test_dates[-1].date()} ({len(pilot_test_dates)} days)")
print(f"Unique products in scope: {len(pilot_products)}")

print("\nWeekly profit comparison (model vs baseline):")
display(weekly_profit_compare)

print("\nOverall profit comparison (full 6 weeks):")
display(overall_profit_compare)

print("\nTotal profit difference (model - baseline):")
print(round(float(overall_profit_compare.loc[0, "profit_diff"]), 2))